# Orientation: Kalman filter

## Overview

The Kalman filter is a widely used estimation algorithm that incorporates a deterministic model of the system, its stochastic/probabilistic characteristics and noisy measurements to estimate a set of state variables over time.  

It has the following components:

- *State vector*: set of parameters that describe the system, which are estimated by the algorithm.
- *Measurement vector*: set of observed measurements taken simultaneously from the system.  
- *System model*: describes how the state vector and the error covariance matrix evolve over time. It's deterministic.
- *Measurement model*: describes how measurements are a function of the state vector. It's deterministic.
- *Noise covariance matrices*: capture randomness on both the system and measurements.
- *Error covariance matrix*: represents the uncertainty in the state vector estimate, and the correlation between the different state variables.

There are a lot of variants of the filter, namely the classical Kalman filter, the extended Kalman filter (EKF), the unscented Kalman filter (UKF), the multiple hypothesis Kalman filter (MHKF), Fast Kalman filter (FKF), etc. It's also closely related to particle filters and other Bayesian filtering techniques. At the same time, one type of filter can vary in its implementation details: order of aproximations, adaptive/fixed covariance matrices, closed-loop/open-loop, sequential/vector measurement update, white/time-correlated noise, etc. 

For the simplest case, the classical Kalman filter, it works under the following assumptions. First, the system dynamics are described by:
$$
\mathbf{\dot{x}}(t) = \mathbf{F}(t) \mathbf{x}(t) + \mathbf{G}(t) \mathbf{w}_s(t)
$$
where:
- $\mathbf{x}(t)$ is the state vector,
- $\mathbf{F}(t)$ is the system matrix,
- $\mathbf{G}(t)$ is the system noise distribution matrix,
- $\mathbf{w}_s(t)$ is the system noise vector, which comprises many independent random nosie sources, each assumed to have a zero-mean symmetric distribution, such as the normal distribution.

Secondly, the measurements are described by:
$$
\mathbf{z}(t) = \mathbf{H}(t) \mathbf{x}(t) + \mathbf{w}_m(t)
$$
where:
- $\mathbf{z}(t)$ is the measurement vector,
- $\mathbf{H}(t)$ is the measurement matrix,
- $\mathbf{w}_m(t)$ is the measurement noise vector, with the same assumptions as the system noise vector.

## Basic algorithm

On a high level, it operates as follows. First, the state is predicted forward from the immediate past using the system model. This, in turn, increases uncertainty. This can be imagined as a normal distribution that's moved and stretched across the space. Secondly, information from the measurements (sensor outputs) is brought in as another distribution and compared against the prediction, which is done by mapping from state space to measurement space through the measurement model and taking the difference between the predicted and actual measurements, known as the innovation. To decide which one to trust more, a variable gain is computed, the Kalman gain, which solves the optimization problem of finding a third distribution that minimizes the covariances. Finally, this gain is used to update both state and the uncertainty. Further visual explanations for this process can be found at [2] and [3].     

This translates into the following steps, as taken from [1]. For each time step $k$, do:

*System propagation*:

1. Calculate the transition matrix, $\mathbf{\Phi}_{k-1}$.

2. Calculate the system noise covariance matrix, $\mathbf{Q}_{k–1}$.

3. Propagate the state vector estimate from $\mathbf{\hat{x}}_{k−1}^+$ to $\mathbf{\hat{x}}_k^−$: 
$$
\mathbf{\hat{x}}_k^{-} = \mathbf{\Phi}_{k-1} \mathbf{\hat{x}}_{k-1}^{+}
$$

4. Propagate the error covariance matrix from $\mathbf{P}_{k–1}^+$ to $\mathbf{P}_k^−$:
$$
\mathbf{P}_k^{-} = \mathbf{\Phi}_{k-1} \mathbf{P}_{k-1}^+ \mathbf{\Phi}_{k-1}^T + \mathbf{Q}_{k-1}
$$

*Measurement update*:

5. Calculate the measurement matrix, $\mathbf{H_k}$.

6. Calculate the measurement noise covariance matrix, $\mathbf{R}_k$.

7. Calculate the Kalman gain matrix, $\mathbf{K}_k$:
$$
\mathbf{K}_k = \mathbf{P}_k^{-} \mathbf{H}_k^T (\mathbf{H}_k \mathbf{P}_k^{-} \mathbf{H}_k^T + \mathbf{R}_k)^{-1}
$$

8. Formulate the measurement, $\mathbf{z}_k$.

9. Update the state vector estimate from $\mathbf{\hat{x}}_k^−$ to $\mathbf{\hat{x}}_k^+$:

$$
\mathbf{\hat{x}}_k^{+} = \mathbf{\hat{x}}_k^{-} + \mathbf{K}_k (\mathbf{\mathbf{z}}_k - \mathbf{H}_k \mathbf{\hat{x}}_k^{-})
= \mathbf{\hat{x}}_k^{-} + \mathbf{K}_k \boldsymbol\delta \mathbf{z}_k^{-}
$$

10. Update the error covariance matrix from $\mathbf{P}_k^−$ to $\mathbf{P}_k^+$:
$$
\mathbf{P}_k^{+} = (\mathbf{I} - \mathbf{K}_k \mathbf{H}_k) \mathbf{P}_k^{-}
$$

The matrices $\mathbf{\Phi}$, $\mathbf{Q}$, $\mathbf{H}$, and $\mathbf{R}$ need to be defined/derived by the user depending on the application, as well as the initial conditions, $\hat{x}_0^+$ and $P_0^+$. For the case of the transition matrix, it can be obtained from the system matrix with a matrix exponential:
$$
\mathbf{\Phi}_{k-1} = e^{\mathbf{F}_{k-1} \Delta t} = \sum_{n=0}^{\infty} \frac{(\mathbf{F}_{k-1} \Delta t)^n}{n!} = \mathbf{I} + \Delta t \mathbf{F}_{k-1} + \frac{\Delta t ^2}{2} \mathbf{F}_{k-1}^2 + \frac{\Delta t ^3}{6} \mathbf{F}_{k-1}^3 + \dots
$$
For the case of the noise covariance matrices, they can be, on the simplest case, diagonal matrices with covariance values based on sensor specifications or empirical observations.

A diagram of the dependencies between the variables is shown below.

<img src="./assets/kf-dependencies.png" alt="Kalman filter dependencies" width="400" height="350">


## Application: quaternion-based orientation estimation

TODO: think what do with gyro biases

### EKF general formulation

The assumption of linear system and measurement models does may not hold well for this application, which motivates the use of the EKF, one of the simplest non-linear alternatives. The EKF linearizes the models through the Jacobian matrix around the current estimate and then feeds them into the standard Kalman filter equations. This means that now the system dynamic model and measurement model are:
$$
\mathbf{\dot{x}}(t) =  \mathbf{f}(\mathbf{x}(t), \mathbf{u}(t), t) + \mathbf{G}(t) \mathbf{w}_s(t) \\
\mathbf{z}(t) = \mathbf{h}(\mathbf{x}(t), t) + \mathbf{w}_m(t)
$$
Where $\mathbf{u}(t)$ is an additional control input to the system.
They are linearized as follows:
$$
\mathbf{F}_{k-1} = \left .\frac{\partial \mathbf{f}}{\partial \mathbf{x}}\right|_{\mathbf{x} = \hat{\mathbf{x}}_{k-1}^{+}, \mathbf{u} = \mathbf{u}_{k-1}, t = t_k} \\
\mathbf{H}_k = \left .\frac{\partial \mathbf{h}}{\partial \mathbf{x}}\right|_{\mathbf{x} = \hat{\mathbf{x}}_k^{-}, t = t_k}
$$

### Transition matrix

For this application, since quarternions have already been explored for the Mahony filter and are a suitable representation for orientation, they will be used inside the state vector. The quaternion differential equation can be expressed as:
$$
\dot{\mathbf{q}} = \frac{1}{2} \mathbf{q} \otimes (0, \mathbf{\omega}) = \frac{1}{2} \boldsymbol\Omega \mathbf{q} = 
\frac{1}{2} \begin{bmatrix}
0 & -\omega_x & -\omega_y & -\omega_z \\
\omega_x & 0 & \omega_z & -\omega_y \\
\omega_y & -\omega_z & 0 & \omega_x \\
\omega_z & \omega_y & -\omega_x & 0
\end{bmatrix} \begin{bmatrix}q_0 \\ q_1 \\ q_2 \\ q_3\end{bmatrix} = 
\begin{bmatrix}
-1/2 \omega_x q_1 - 1/2 \omega_y q_2 - 1/2 \omega_z q_3 \\
1/2 \omega_x q_0 + 1/2 \omega_z q_2 - 1/2 \omega_y q_3 \\
1/2 \omega_y q_0 - 1/2 \omega_z q_1 + 1/2 \omega_x q_3 \\
1/2 \omega_z q_0 + 1/2 \omega_y q_1 - 1/2 \omega_x q_2
\end{bmatrix}
$$
And can be viewed as the continuous-time system model, $\mathbf{f}(\mathbf{q}(t), \bm{\omega}(t), t)$. Computing the Jacobian:
$$
\mathbf{F} = \frac{\partial \mathbf{f}(\mathbf{q}, \boldsymbol\omega)}{\partial \mathbf{q}} = \begin{bmatrix}
\frac{\partial\mathbf{f}(\mathbf{q}, \boldsymbol\omega)}{\partial q_0} &
\frac{\partial\mathbf{f}(\mathbf{q}, \boldsymbol\omega)}{\partial q_1} &
\frac{\partial\mathbf{f}(\mathbf{q}, \boldsymbol\omega)}{\partial q_2} &
\frac{\partial\mathbf{f}(\mathbf{q}, \boldsymbol\omega)}{\partial q_3}
\end{bmatrix} = \begin{bmatrix}
0 & -1/2 \omega_x & -1/2 \omega_y & -1/2 \omega_z  \\
1/2 \omega_x & 0 & 1/2 \omega_z & -1/2 \omega_y \\
1/2 \omega_y & -1/2 \omega_z & 0 & 1/2 \omega_x \\
1/2 \omega_z & 1/2 \omega_y & -1/2 \omega_x & 0
\end{bmatrix}
$$
Lastly, with a first order approximation, the transition matrix can be expressed as:
$$
\mathbf{\Phi}_{k-1} = e^{\mathbf{F}_{k-1} \Delta t} \approx \mathbf{I}_4 + \frac{\Delta t}{2} \mathbf{F}_{k-1} = \begin{bmatrix}
1 & -\frac{\Delta t}{2} \omega_x & -\frac{\Delta t}{2} \omega_y & -\frac{\Delta t}{2} \omega_z \\
\frac{\Delta t}{2} \omega_x & 1 & \frac{\Delta t}{2} \omega_z & -\frac{\Delta t}{2} \omega_y \\
\frac{\Delta t}{2} \omega_y & -\frac{\Delta t}{2} \omega_z & 1 & \frac{\Delta t}{2} \omega_x \\
\frac{\Delta t}{2} \omega_z & \frac{\Delta t}{2} \omega_y & -\frac{\Delta t}{2} \omega_x & 1
\end{bmatrix}
$$

### Measurement matrix

### System noise covariance matrix

### Measurement noise covariance matrix

---

$$
\mathbf{Q}_t = \sigma_{\boldsymbol\omega}^2\mathbf{W}_t\mathbf{W}_t^T \\
$$
$$
\begin{split}\begin{array}{rcl}
\mathbf{W}_t &=& \frac{\partial\mathbf{f}(\mathbf{q}_{t-1}, \boldsymbol\omega_t)}{\partial\boldsymbol\omega} \\
&=& \begin{bmatrix}
\frac{\partial\mathbf{f}(\mathbf{q}_{t-1}, \boldsymbol\omega_t)}{\partial\omega_x} &
\frac{\partial\mathbf{f}(\mathbf{q}_{t-1}, \boldsymbol\omega_t)}{\partial\omega_y} &
\frac{\partial\mathbf{f}(\mathbf{q}_{t-1}, \boldsymbol\omega_t)}{\partial\omega_z}
\end{bmatrix} \\
&=& \frac{\Delta t}{2}
\begin{bmatrix}
-q_x & -q_y & -q_z \\
q_w & -q_z & q_y \\
q_z & q_w & -q_x \\
-q_y & q_x & q_w
\end{bmatrix}
\end{array}\end{split}
$$

$$
\begin{split}\mathbf{z}_t = \begin{bmatrix}\mathbf{a}_t \\ \mathbf{m}_t\end{bmatrix} =
\begin{bmatrix}a_x \\ a_y \\ a_z \\ m_x \\ m_y \\ m_z\end{bmatrix}\end{split}
$$


$$
\begin{split}\begin{array}{rcl}
\mathbf{h}(\hat{\mathbf{q}}_t) &=& \begin{bmatrix}\hat{\mathbf{a}} \\ \hat{\mathbf{m}}\end{bmatrix}
= \begin{bmatrix}\mathbf{C}(\hat{\mathbf{q}})^T \mathbf{g} \\ \mathbf{C}(\hat{\mathbf{q}})^T \mathbf{r}\end{bmatrix} \\
&=&
2 \begin{bmatrix}
g_x (\frac{1}{2} - q_y^2 - q_z^2) + g_y (q_wq_z + q_xq_y) + g_z (q_xq_z - q_wq_y) \\
g_x (q_xq_y - q_wq_z) + g_y (\frac{1}{2} - q_x^2 -  q_z^2) + g_z (q_wq_x + q_yq_z) \\
g_x (q_wq_y + q_xq_z) + g_y (q_yq_z - q_wq_x) + g_z (\frac{1}{2} - q_x^2 -  q_y^2) \\
r_x (\frac{1}{2} - q_y^2 - q_z^2) + r_y (q_wq_z + q_xq_y) + r_z (q_xq_z - q_wq_y) \\
r_x (q_xq_y - q_wq_z) + r_y (\frac{1}{2} - q_x^2 -  q_z^2) + r_z (q_wq_x + q_yq_z) \\
r_x (q_wq_y + q_xq_z) + r_y (q_yq_z - q_wq_x) + r_z (\frac{1}{2} - q_x^2 -  q_y^2)
\end{bmatrix}
\end{array}\end{split}
$$

$$
\begin{split}\begin{array}{rcl}
\mathbf{H}(\hat{\mathbf{q}}_t) &=& \frac{\partial\mathbf{h}(\hat{\mathbf{q}}_t)}{\partial\mathbf{q}} \\
&=&\begin{bmatrix}
\frac{\partial\mathbf{h}(\hat{\mathbf{q}}_t)}{\partial q_w} &
\frac{\partial\mathbf{h}(\hat{\mathbf{q}}_t)}{\partial q_x} &
\frac{\partial\mathbf{h}(\hat{\mathbf{q}}_t)}{\partial q_y} &
\frac{\partial\mathbf{h}(\hat{\mathbf{q}}_t)}{\partial q_z}
\end{bmatrix} \\
&=&
2
\begin{bmatrix}
g_xq_w + g_yq_z - g_zq_y & g_xq_x + g_yq_y + g_zq_z & -g_xq_y + g_yq_x - g_zq_w & -g_xq_z + g_yq_w + g_zq_x \\
-g_xq_z + g_yq_w + g_zq_x & g_xq_y - g_yq_x + g_zq_w &  g_xq_x + g_yq_y + g_zq_z & -g_xq_w - g_yq_z + g_zq_y \\
g_xq_y - g_yq_x + g_zq_w & g_xq_z - g_yq_w - g_zq_x &  g_xq_w + g_yq_z - g_zq_y &  g_xq_x + g_yq_y + g_zq_z \\
r_xq_w + r_yq_z - r_zq_y & r_xq_x + r_yq_y + r_zq_z & -r_xq_y + r_yq_x - r_zq_w & -r_xq_z + r_yq_w + r_zq_x \\
-r_xq_z + r_yq_w + r_zq_x & r_xq_y - r_yq_x + r_zq_w &  r_xq_x + r_yq_y + r_zq_z & -r_xq_w - r_yq_z + r_zq_y \\
r_xq_y - r_yq_x + r_zq_w & r_xq_z - r_yq_w - r_zq_x &  r_xq_w + r_yq_z - r_zq_y &  r_xq_x + r_yq_y + r_zq_z
\end{bmatrix}
\end{array}\end{split}
$$


$$
\begin{split}\mathbf{R} =
\begin{bmatrix}
\boldsymbol\sigma_\mathbf{a}^2\mathbf{I}_3 & \mathbf{0}_3 \\
\mathbf{0}_3 & \boldsymbol\sigma_\mathbf{m}^2\mathbf{I}_3
\end{bmatrix}\end{split}
$$

$$
\mathbf{q}_t \leftarrow \frac{1}{\|\mathbf{q}_t\|}\mathbf{q}_t
$$

## Sources:

1. [Principles of GNSS, Inertial, and Multisensor Integrated Navigation Systems - Paul D. Groves - Chapter 3](../../docs/assets/books/Principles%20of%20GNSS,%20Inertial,%20and%20Multisensor%20Integrated%20Navigation%20Systems%20-%20Paul%20Groves.pdf)
2. [Why Kalman Filters Work So Well](https://www.youtube.com/watch?v=zFmO_b05jU4)
3. [Kalman Filter - VISUALLY EXPLAINED!](https://www.youtube.com/watch?v=-DiZGpAh7T4)
4. [Wikipedia: Matrix Exponential](https://en.wikipedia.org/wiki/Matrix_exponential)
5. [AHRS: Fast Kalman Filter Documentation](https://ahrs.readthedocs.io/en/latest/filters/fkf.html#)
6. [AHRS: Extended Kalman Filter Documentation](https://ahrs.readthedocs.io/en/latest/filters/ekf.html#)
7. [AHRS: Unscented Kalman Filter Documentation](https://ahrs.readthedocs.io/en/latest/filters/ukf.html#)